# Day 3 lab, Parts A and B: train and convert a motion classifier

**Goal.** You calculate features from your Day 2 dataset, train a small
classifier, convert it to the LiteRT format, and write the files that the
sketch for the board needs.

**Deliverable.** Three files in `sketches/motion_classifier/`: `model.h`,
`model_settings.h`, and `test_window.h`. The numbers of this notebook go into
`report.md`.

**Time.** Part A: 45 min, sections 0 to 6. Part B: about 10 min of its
50 min, sections 7 and 8. The complete notebook runs in about one minute.

**How to work.** Run the cells in order. A cell with the mark
`TODO (student)` has a task. The notebook has five tasks. The cell after a
task checks your result.

## 0. Setup

Run this cell first. It prints the versions that you write in your report,
and it selects the dataset.

- If the folder `Labs/day02/data/` has recordings, the notebook uses them.
- If not, the notebook uses the fallback dataset of Day 2 in
  `Labs/day02/fallback_data/`. The fallback signals are simulated. Nobody
  recorded them. A model that learns from them does not work on the real
  kit.

In [ ]:
import json
import os
import platform
import sys

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # no start messages of TensorFlow

import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter

tf.get_logger().setLevel("ERROR")            # no INFO lines of the converter

print("python    ", sys.version.split()[0])
print("numpy     ", np.__version__)
print("tensorflow", tf.__version__)
print("keras     ", keras.__version__)
print("machine   ", platform.machine())

RATE_HZ = 50
WINDOW = 100          # samples: 2 s at 50 Hz
STRIDE = 10           # samples: 0.2 s
FFT_LENGTH = 32
CLASSES = ["idle", "terrestrial", "lift", "maritime"]

# The lab folder. The notebook of the folder solutions/ uses the folder above.
LAB = "." if os.path.isdir("sketches") and os.path.isdir("host") else ".."
DAY2 = os.path.join(LAB, "..", "day02")
SKETCH = os.path.join(LAB, "sketches", "motion_classifier")
sys.path.insert(0, os.path.join(LAB, "host"))


def recording_files(folder):
    '''Return the names of the recording files in a folder.'''
    if not os.path.isdir(folder):
        return []
    return sorted(name for name in os.listdir(folder)
                  if name.endswith(".csv") and len(name.split(".")) == 4)


DATA = os.path.join(DAY2, "data")
if not recording_files(DATA):
    DATA = os.path.join(DAY2, "fallback_data")
    if not recording_files(DATA):
        sys.path.insert(0, os.path.join(DAY2, "host"))
        import make_fallback_dataset
        import motion_sim
        os.makedirs(DATA, exist_ok=True)
        for label in motion_sim.CLASSES:
            for session in make_fallback_dataset.SESSIONS:
                for number in range(1, make_fallback_dataset.RECORDINGS + 1):
                    make_fallback_dataset.write_recording(DATA, label, session,
                                                          number)
    print()
    print("No recording in Labs/day02/data/. The notebook uses the fallback")
    print("dataset. Its signals are simulated. Nobody recorded them.")
print()
print("data folder:", os.path.normpath(DATA), "with",
      len(recording_files(DATA)), "files")

## 1. The dataset and the split (3 min)

The code loads the recordings of Day 2. It uses the three acceleration axes
in $g$, as the kit lab does.

The file `split.json` of Day 2 names the test sessions. If the file does not
exist, the last session is the test session.

You do not change this code.

In [ ]:
def load_recording(folder, name):
    '''Return one recording: label, session, and the samples of ax, ay, az.'''
    label, session, _, _ = name.split(".")
    rows = []
    with open(os.path.join(folder, name), encoding="ascii",
              errors="replace") as handle:
        for line in handle:
            fields = line.strip().split(",")
            if line.startswith("#") or len(fields) != 8 or fields[0] == "n":
                continue
            rows.append([float(f) for f in fields[2:5]])
    return {"name": name, "label": label, "session": session,
            "data": np.array(rows, dtype=np.float32)}


def cut_windows(record):
    '''Return the windows of one recording: array (windows, 100, 3).'''
    data = record["data"]
    starts = range(0, len(data) - WINDOW + 1, STRIDE)
    return np.stack([data[s:s + WINDOW] for s in starts])


records = [load_recording(DATA, name) for name in recording_files(DATA)]
labels = [c for c in CLASSES if any(r["label"] == c for r in records)]
labels += sorted({r["label"] for r in records} - set(labels))
sessions = sorted({r["session"] for r in records})

split_file = os.path.join(DATA, "split.json")
if os.path.exists(split_file):
    with open(split_file, encoding="utf-8") as handle:
        TEST_SESSIONS = json.load(handle)["test_sessions"]
    print("split.json: test sessions", TEST_SESSIONS)
else:
    TEST_SESSIONS = [sessions[-1]]
    print("no split.json: the last session is the test session", TEST_SESSIONS)


def windows_of(session_list):
    '''Return (windows, class numbers) of all recordings of the sessions.'''
    group = [r for r in records if r["session"] in session_list]
    x = np.concatenate([cut_windows(r) for r in group])
    y = np.concatenate([[labels.index(r["label"])] * len(cut_windows(r))
                        for r in group])
    return x, y


train_sessions = [s for s in sessions if s not in TEST_SESSIONS]
w_train, y_train = windows_of(train_sessions)
w_test, y_test = windows_of(TEST_SESSIONS)

print(f"classes:  {labels}")
print(f"training: sessions {train_sessions}, {len(w_train)} windows")
print(f"test:     sessions {TEST_SESSIONS}, {len(w_test)} windows")
print(f"one window: {w_train.shape[1]} samples x {w_train.shape[2]} axes = "
      f"{w_train.shape[1] * w_train.shape[2]} values")

## 2. Features (12 min)

Part 3 of the lecture gave the feature block. For each axis, the code removes
the mean of the window and calculates 21 features:

| Position | Feature |
|---|---|
| 0 | RMS |
| 1, 2 | skewness and kurtosis of the samples |
| 3, 4 | skewness and kurtosis of the 16 power values |
| 5 to 20 | spectral power of the frequency bins 1 to 16 |

Three axes give $3 \times 21 = 63$ features.

**Task 1.** Write the function `remove_mean_and_rms`. It gets the samples of
one axis. It returns two results: the samples with the mean removed, and the
RMS of these samples.

$$x_{\mathrm{RMS}} = \sqrt{\frac{1}{n}\left(x_1^2 + x_2^2 + \dots + x_n^2\right)}$$

In [ ]:
def remove_mean_and_rms(x):
    '''Return (x with its mean removed, RMS of the result).'''
    # TODO (student): replace the two lines below.
    centred = x
    rms = 0.0
    return centred, rms

In [ ]:
def check_task_1():
    '''Compare remove_mean_and_rms with two known cases.'''
    t = np.arange(100) / RATE_HZ
    sine = (1.0 + np.sin(2 * np.pi * 2.0 * t)).astype(np.float32)   # 4 cycles
    centred, rms = remove_mean_and_rms(sine)
    constant = np.full(100, 0.98, dtype=np.float32)
    _, rms_constant = remove_mean_and_rms(constant)
    correct = (abs(float(np.mean(centred))) < 1e-5
               and abs(float(rms) - 1 / np.sqrt(2)) < 1e-3
               and abs(float(rms_constant)) < 1e-5)
    print("Task 1:", "complete" if correct else "not complete")
    return correct


task_1_complete = check_task_1()

The next cell has the other feature functions. Read them. The file
`sketches/motion_classifier/motion_features.h` has the same calculation in C++ for
the board.

In [ ]:
def moments(x):
    '''Return (skewness, kurtosis) of the values. 0 for a constant signal.'''
    d = x - x.mean()
    m2 = np.mean(d ** 2)
    if m2 < 1e-12:
        return 0.0, 0.0
    return np.mean(d ** 3) / m2 ** 1.5, np.mean(d ** 4) / m2 ** 2 - 3.0


def spectral_power(x, nfft=FFT_LENGTH):
    '''Largest power of each frequency bin over the frames of the window.'''
    power = np.zeros(nfft // 2 + 1)
    for start in range(0, len(x), nfft):
        frame = x[start:start + nfft]                  # the last frame is short
        spectrum = np.abs(np.fft.rfft(frame, nfft))    # rfft adds zeros
        power = np.maximum(power, spectrum ** 2 / nfft)
    return power[1:]                                   # no bin at 0 Hz


def axis_features(x):
    '''Return the 21 features of one axis.'''
    centred, rms = remove_mean_and_rms(x)
    skewness, kurtosis = moments(centred)
    power = spectral_power(centred)
    power_skewness, power_kurtosis = moments(power)
    return np.concatenate([[rms, skewness, kurtosis, power_skewness,
                            power_kurtosis], power])


def extract_features(window):
    '''Return the 63 features of one window with the shape (100, 3).'''
    return np.concatenate([axis_features(window[:, k])
                           for k in range(window.shape[1])]).astype(np.float32)


f_train = np.array([extract_features(w) for w in w_train])
f_test = np.array([extract_features(w) for w in w_test])
N_FEATURES = f_train.shape[1]
print("feature matrix of the training set:", f_train.shape)
print("feature matrix of the test set:    ", f_test.shape)

The plot shows the spectral power of the first training window of each class,
for each axis. The frequency of bin $k$ is $k \times 50 / 32 = k \times 1.56$
Hz. Look for the axis and the bin that separate the classes.

In [ ]:
figure, axes = plt.subplots(1, len(labels), figsize=(3.6 * len(labels), 2.8),
                            sharey=True)
bins = np.arange(1, FFT_LENGTH // 2 + 1) * RATE_HZ / FFT_LENGTH
for axis, label in zip(np.atleast_1d(axes), labels):
    index = int(np.argmax(y_train == labels.index(label)))
    for k, name in enumerate(("ax", "ay", "az")):
        power = f_train[index, k * 21 + 5:(k + 1) * 21]
        axis.semilogy(bins, np.maximum(power, 1e-8), marker=".", label=name)
    axis.set_title(label, fontsize=9)
    axis.set_xlabel("frequency (Hz)")
    axis.grid(alpha=0.3)
np.atleast_1d(axes)[0].set_ylabel("spectral power")
np.atleast_1d(axes)[-1].legend(fontsize=7)
figure.tight_layout()
plt.show()

## 3. Normalize the features (2 min)

The features have very different sizes. The network trains better when each
input has a mean of 0 and a standard deviation of 1.

The mean and the standard deviation come from the **training set only**
(Day 2: no leakage). The board must use the same 63 means and the same 63
standard deviations. Section 8 writes them into the file
`model_settings.h`.

In [ ]:
feature_mean = f_train.mean(axis=0)
feature_std = np.maximum(f_train.std(axis=0), 1e-6)

x_train = (f_train - feature_mean) / feature_std
x_test = (f_test - feature_mean) / feature_std
print("training input: mean %.3f, standard deviation %.3f"
      % (x_train.mean(), x_train.std()))
print("test input:     mean %.3f, standard deviation %.3f"
      % (x_test.mean(), x_test.std()))

## 4. The model (8 min)

The classifier of the kit lab is a small fully connected network:

| Layer | Neurons | Activation |
|---|---|---|
| Input | 63 features | none |
| Hidden layer 1 | 20 | ReLU |
| Hidden layer 2 | 10 | ReLU |
| Output | one for each class: 4 | softmax |

**Task 2.** Complete the function `build_model`. Add the two hidden layers.
The function gets the number of inputs, so that section 6 can use it for a
different input.

In [ ]:
def build_model(inputs, classes):
    '''Return the classifier: inputs, 20, 10, classes.'''
    return keras.Sequential([
        keras.Input(shape=(inputs,)),
        # TODO (student): add the two hidden layers here.
        keras.layers.Dense(classes, activation="softmax"),
    ])

In [ ]:
keras.utils.set_random_seed(0)
model = build_model(N_FEATURES, len(labels))
model.summary()

expected = (N_FEATURES * 20 + 20) + (20 * 10 + 10) + (10 * len(labels) + len(labels))
task_2_complete = model.count_params() == expected
print()
print(f"parameters: {model.count_params()} (expected: {expected})")
print("Task 2:", "complete" if task_2_complete else "not complete")

## 5. Train and test (8 min)

The training needs some seconds on a laptop CPU. The test set is the test
session of Day 2. The model did not see it during the training.

In [ ]:
def train(model, x, y, epochs=30):
    '''Train a model and return the history of the loss.'''
    model.compile(optimizer=keras.optimizers.Adam(0.005),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model.fit(x, y, epochs=epochs, batch_size=32, verbose=0)


def confusion(model, x, y):
    '''Return the confusion matrix: rows are real classes.'''
    predicted = model.predict(x, verbose=0).argmax(axis=1)
    matrix = np.zeros((len(labels), len(labels)), dtype=int)
    for true, guess in zip(y, predicted):
        matrix[true, guess] += 1
    return matrix


history = train(model, x_train, y_train)
accuracy_train = model.evaluate(x_train, y_train, verbose=0)[1]
accuracy_test = model.evaluate(x_test, y_test, verbose=0)[1]
print(f"loss after the first epoch: {history.history['loss'][0]:.3f}")
print(f"loss after the last epoch:  {history.history['loss'][-1]:.3f}")
print(f"training accuracy: {100 * accuracy_train:.1f} percent")
print(f"test accuracy:     {100 * accuracy_test:.1f} percent "
      f"(test sessions {TEST_SESSIONS})")
print()
print(f"{'real class':<14}" + "".join(f"{label[:6]:>8}" for label in labels))
for label, row in zip(labels, confusion(model, x_test, y_test)):
    print(f"{label:<14}" + "".join(f"{value:>8}" for value in row))

## 6. Features against the raw window (10 min)

Part 3 of the lecture compared two inputs: the 63 features, and the raw
window with $100 \times 3 = 300$ values. You now train the two models. Each
model trains one time for each session: that session is the test session,
and the other sessions are the training set.

**Task 3.** Write your prediction **before** you run the experiment. The
prediction is part of the assessment. A wrong prediction costs no points. A
missing prediction does.

In [ ]:
# TODO (student): replace each "?".
prediction = {
    # a number
    "parameters of the model with the raw window": "?",
    # "features", "raw window", or "equal"
    "the input with the higher test accuracy": "?",
    # one sentence
    "the reason": "?",
}
for question, answer in prediction.items():
    print(f"{question:<46} {answer}")

In [ ]:
def experiment(raw):
    '''Train one model for each test session. Return the list of results.'''
    results = []
    for session in sessions:
        others = [s for s in sessions if s != session]
        if not others:
            continue
        wa, ya = windows_of(others)
        wb, yb = windows_of([session])
        if raw:
            xa, xb = wa.reshape(len(wa), -1), wb.reshape(len(wb), -1)
        else:
            xa = np.array([extract_features(w) for w in wa])
            xb = np.array([extract_features(w) for w in wb])
        mean, std = xa.mean(axis=0), np.maximum(xa.std(axis=0), 1e-6)
        keras.utils.set_random_seed(0)
        net = build_model(xa.shape[1], len(labels))
        train(net, (xa - mean) / std, ya)
        accuracy = net.evaluate((xb - mean) / std, yb, verbose=0)[1]
        results.append((session, net.count_params(), accuracy))
    return results


results_features = experiment(raw=False)
results_raw = experiment(raw=True)

print(f"{'input':<14}{'values':>8}{'parameters':>12}"
      + "".join(f"{'test ' + s:>10}" for s, _, _ in results_features)
      + f"{'mean':>8}")
for name, values, results in (("features", N_FEATURES, results_features),
                              ("raw window", WINDOW * 3, results_raw)):
    if not results:
        continue
    accuracies = [a for _, _, a in results]
    print(f"{name:<14}{values:>8}{results[0][1]:>12}"
          + "".join(f"{100 * a:>9.1f}%" for a in accuracies)
          + f"{100 * np.mean(accuracies):>7.1f}%")

> TODO (student): answer these three questions in `report.md`.
>
> 1. Compare the result with your prediction of task 3.
> 2. Which input needs less flash on the board? Which input needs less RAM?
>    Give the numbers.
> 3. The raw window needs no feature code on the board. Is this a reason to
>    select it? Use your accuracy numbers.

## 7. Convert to LiteRT (5 min)

This section is the first step of Part B.

**Task 4.** Convert the feature model of section 5 to the LiteRT format. Use
`tf.lite.TFLiteConverter.from_keras_model` and its method `convert`, as Part 1
of the lecture showed. The result is an object of the type `bytes`.

In [ ]:
# TODO (student): replace the line below with the two lines of the converter.
tflite_model = b""

In [ ]:
task_4_complete = len(tflite_model) > 0 and tflite_model[4:8] == b"TFL3"
print("Task 4:", "complete" if task_4_complete else "not complete")

if task_4_complete:
    interpreter = Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]

    def litert_predict(x):
        '''Run the LiteRT model for each row of x.'''
        result = []
        for row in x:
            interpreter.set_tensor(inp["index"], row[None, :].astype(np.float32))
            interpreter.invoke()
            result.append(interpreter.get_tensor(out["index"])[0].copy())
        return np.array(result)

    y_litert = litert_predict(x_test)
    y_keras = model.predict(x_test, verbose=0)
    operators = [op["op_name"] for op in interpreter._get_ops_details()
                 if op["op_name"] != "DELEGATE"]
    weights = model.count_params() * 4
    print()
    print(f"LiteRT file:            {len(tflite_model)} bytes")
    print(f"weights in float32:     {weights} bytes "
          f"({100 * weights / len(tflite_model):.0f} percent of the file)")
    print(f"input:                  shape {[int(v) for v in inp['shape']]}, "
          f"type {np.dtype(inp['dtype']).name}")
    print(f"output:                 shape {[int(v) for v in out['shape']]}, "
          f"type {np.dtype(out['dtype']).name}")
    print(f"operators:              {operators}")
    print(f"operator types:         {sorted(set(operators))}")
    print(f"largest difference between Keras and LiteRT: "
          f"{np.abs(y_litert - y_keras).max():.2e}")
    print(f"test accuracy of the LiteRT model: "
          f"{100 * (y_litert.argmax(axis=1) == y_test).mean():.1f} percent")

The operator types of the output above are the operators that your sketch
must name in its operator resolver (task B1 of the lab).

## 8. Write the files for the sketch (5 min)

The next cell writes three files into `sketches/motion_classifier/`:

| File | Content |
|---|---|
| `model.h` | The LiteRT file as a C array |
| `model_settings.h` | The class names, and the mean and the standard deviation of each feature |
| `test_window.h` | One test window, its 63 features, and the output of the model for it |

The sketch uses `test_window.h` for a self-test at the start: it calculates
the features of the test window on the board and compares them with the
values of this notebook.

In [ ]:
def c_float(value):
    '''Return a number as a C float constant, for example 1.0f or 2.5e-07f.'''
    text = f"{float(value):.9g}"
    if "." not in text and "e" not in text:
        text += ".0"
    return text + "f"


def c_array(name, values, per_line=6):
    '''Return the C text of a constant float array.'''
    lines = [f"const float {name}[{len(values)}] = {{"]
    for start in range(0, len(values), per_line):
        chunk = values[start:start + per_line]
        lines.append("    " + ", ".join(c_float(v) for v in chunk) + ",")
    lines.append("};")
    return "\n".join(lines)


def write_sketch_files():
    '''Write model.h, model_settings.h, and test_window.h.'''
    import tflite_to_header                 # Labs/day03/host/

    os.makedirs(SKETCH, exist_ok=True)
    note = ("Made by motion_classifier.ipynb. Do not change this file by "
            "hand.\n// Dataset: " + os.path.basename(os.path.normpath(DATA)))

    with open(os.path.join(SKETCH, "model.h"), "w", encoding="ascii") as handle:
        handle.write(tflite_to_header.make_header(tflite_model, "model.tflite"))

    names = ", ".join(f'"{label}"' for label in labels)
    with open(os.path.join(SKETCH, "model_settings.h"), "w",
              encoding="ascii") as handle:
        handle.write(f"// model_settings.h - settings of the motion classifier\n"
                     f"// {note}\n\n"
                     "#ifndef MODEL_SETTINGS_H_\n#define MODEL_SETTINGS_H_\n\n"
                     f"constexpr int kNumClasses = {len(labels)};\n"
                     f"const char* const kClassNames[kNumClasses] = {{{names}}};\n\n"
                     "// Mean and standard deviation of each feature in the "
                     "training set.\n"
                     + c_array("kFeatureMean", feature_mean) + "\n\n"
                     + c_array("kFeatureStd", feature_std) + "\n\n"
                     "#endif  // MODEL_SETTINGS_H_\n")

    # The test window: the first test window of the last class.
    index = int(np.argmax(y_test == y_test.max()))
    window = w_test[index]
    features = extract_features(window)
    output = litert_predict(((features - feature_mean) / feature_std)[None, :])[0]
    with open(os.path.join(SKETCH, "test_window.h"), "w",
              encoding="ascii") as handle:
        handle.write("// test_window.h - one window with its known results\n"
                     f"// {note}\n\n"
                     "#ifndef TEST_WINDOW_H_\n#define TEST_WINDOW_H_\n\n"
                     f"// The real class of the window: {labels[y_test[index]]}\n"
                     f"constexpr int kTestClass = {int(y_test[index])};\n\n"
                     "// 100 samples of ax, ay, az in g: sample 0, sample 1, ...\n"
                     + c_array("kTestWindow", window.reshape(-1)) + "\n\n"
                     "// The 63 features of the window, before the "
                     "normalization.\n"
                     + c_array("kTestFeatures", features) + "\n\n"
                     "// The output of the LiteRT model on the laptop.\n"
                     + c_array("kTestOutput", output) + "\n\n"
                     "#endif  // TEST_WINDOW_H_\n")
    return labels[y_test[index]], output


if task_4_complete:
    test_label, test_output = write_sketch_files()
    for name in ("model.h", "model_settings.h", "test_window.h"):
        path = os.path.join(SKETCH, name)
        print(f"wrote {os.path.normpath(path)} ({os.path.getsize(path)} bytes)")
    print()
    print(f"test window: class '{test_label}', output of the model "
          + ", ".join(f"{v:.3f}" for v in test_output))
else:
    print("Task 4 is not complete. The notebook did not write the files.")

**Task 5.** Estimate the arena size before you measure it on the board. The
next cell gives the size of each tensor of the model. Use the rule of Part 2
of the lecture: the peak is the largest sum of the input and the output of
one operator. Write three numbers in `report.md`:

1. the peak of the activations in bytes,
2. the arena size that you try first,
3. after Part C: the value of `arena_used_bytes()` on the board.

In [ ]:
if task_4_complete:
    details = {d["index"]: d for d in interpreter.get_tensor_details()}
    print(f"{'operator':<18}{'input bytes':>12}{'output bytes':>14}{'sum':>8}")
    for op in interpreter._get_ops_details():
        if op["op_name"] == "DELEGATE":
            continue
        size = lambda i: int(np.prod(details[i]["shape"])) * np.dtype(
            details[i]["dtype"]).itemsize
        first_input = size(op["inputs"][0])
        output = size(op["outputs"][0])
        print(f"{op['op_name']:<18}{first_input:>12}{output:>14}"
              f"{first_input + output:>8}")
else:
    print("Task 4 is not complete.")

> TODO (student): write your three numbers of task 5 in `report.md`.

## Summary

- The features make the input smaller: 63 values in place of 300.
- The mean and the standard deviation of the training set are a part of the
  model. The board needs them.
- The LiteRT file holds the weights and four operators. The same file goes
  to the board as a C array.
- Continue with Part B of the lab: open `sketches/motion_classifier/` in the
  Arduino IDE.